### ============================================
### NOTEBOOK : 04_pubmedbert_extraction.ipynb
### PARTIE 1 : Installation
### ============================================

In [1]:
import sys
print(sys.executable)

c:\Users\daouc\MedicAIde\m-dic-AIde\.venv\Scripts\python.exe


In [2]:
import sys
!{sys.executable} -m pip install transformers torch datasets

   ---------------------------------------- 0.0/10.4 MB ? eta -:--:--
   ---------------- ----------------------- 4.2/10.4 MB 23.8 MB/s eta 0:00:01
   ---------------------------------------- 10.4/10.4 MB 26.4 MB/s eta 0:00:00
   ---------------------------------------- 0.0/113.8 MB ? eta -:--:--
   --- ------------------------------------ 8.9/113.8 MB 44.3 MB/s eta 0:00:03
   ----- ---------------------------------- 16.8/113.8 MB 41.4 MB/s eta 0:00:03
   --------- ------------------------------ 26.7/113.8 MB 43.0 MB/s eta 0:00:03
   ------------ --------------------------- 35.9/113.8 MB 42.8 MB/s eta 0:00:02
   --------------- ------------------------ 44.8/113.8 MB 42.6 MB/s eta 0:00:02
   ------------------ --------------------- 54.0/113.8 MB 42.8 MB/s eta 0:00:02
   -------------------- ------------------- 59.2/113.8 MB 40.2 MB/s eta 0:00:02
   ----------------------- ---------------- 66.8/113.8 MB 39.5 MB/s eta 0:00:02
   -------------------------- ------------- 76.5/113.8 MB 40.2 


[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
import sys
import transformers
import torch

In [4]:
print(f"\n Versions :")
print(f"  - transformers : {transformers.__version__}")
print(f"  - torch : {torch.__version__}")
print(f"  - CUDA disponible : {torch.cuda.is_available()}")


 Versions :
  - transformers : 5.2.0
  - torch : 2.10.0+cpu
  - CUDA disponible : False


### ============================================
### PARTIE 2 : Test sur 1 exemple
### ============================================


In [5]:
from transformers import AutoTokenizer, AutoModel
import torch

print("\n" + "=" * 60)
print(" TEST PubMedBERT SUR 1 EXEMPLE")
print("=" * 60)



 TEST PubMedBERT SUR 1 EXEMPLE


In [6]:
print("chargement du modèle PubMedBERT...")
model_name = "microsoft/BiomedNLP-PubMedBERT-base-uncased-abstract-fulltext"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)
print(f" Modèle chargé : {model_name}")

chargement du modèle PubMedBERT...


config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

c:\Users\daouc\MedicAIde\m-dic-AIde\.venv\Lib\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\daouc\.cache\huggingface\hub\models--microsoft--BiomedNLP-PubMedBERT-base-uncased-abstract-fulltext. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


tokenizer_config.json:   0%|          | 0.00/28.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: microsoft/BiomedNLP-PubMedBERT-base-uncased-abstract-fulltext
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


 Modèle chargé : microsoft/BiomedNLP-PubMedBERT-base-uncased-abstract-fulltext


### --------------------------------------------
### 2. TESTER SUR 1 NOTE
### --------------------------------------------


In [7]:
# Charger une note d'exemple 
import pandas as pd
train_data=pd.read_csv("../data/processed/train_data.csv")
exemple=train_data.iloc[0]
print(f"\n Note test :")
print(f"  - SUBJECT_ID : {exemple['SUBJECT_ID']}")
print(f"  - Longueur : {exemple['text_length']} caractères")
print(f"  - Persona : {exemple['persona_target']}")


 Note test :
  - SUBJECT_ID : 16143
  - Longueur : 184 caractères
  - Persona : Urgentiste


In [8]:
# Prendre les 512 premiers caractères (limite BERT)
text = exemple['TEXT_CLEAN'][:512]
print(f"\n Texte à analyser (512 premiers car) :")
print(text[:200] + "...\n")


 Texte à analyser (512 premiers car) :
Sinus rhythm Marked left axis deviation Poor R wave progression and Q -waves in leads V1-V2 consistent with old septal infarct Nonspecific lateral ST-T wave changes No previous tracing...



### --------------------------------------------
### 3. TOKENIZATION
### --------------------------------------------

In [9]:
print(" Tokenization...")

inputs = tokenizer(
    text,
    return_tensors="pt",  # Format PyTorch
    truncation=True,
    max_length=512,
    padding=True
)

print(f" Tokens : {inputs['input_ids'].shape}")

 Tokenization...
 Tokens : torch.Size([1, 36])


### --------------------------------------------
### 4. EXTRACTION EMBEDDINGS
### --------------------------------------------

In [10]:
print("\nExtraction embeddings...")

with torch.no_grad():  # Pas besoin de gradient (inference)
    outputs = model(**inputs)

# Récupérer embeddings
embeddings = outputs.last_hidden_state  # [1, seq_len, 768]
pooled_embedding = embeddings.mean(dim=1)  # [1, 768] (moyenne)

print(f" Embeddings extraits : {pooled_embedding.shape}")
print(f"   Exemple valeurs : {pooled_embedding[0][:10]}")



Extraction embeddings...
 Embeddings extraits : torch.Size([1, 768])
   Exemple valeurs : tensor([-0.1691,  0.1931,  0.0453, -0.1298,  0.0333,  0.1255,  0.0522,  0.1427,
         0.2985, -0.0140])


### --------------------------------------------
### 5. VÉRIFICATION
### --------------------------------------------


In [11]:
print("\n TEST RÉUSSI !")
print(f"   PubMedBERT fonctionne correctement sur le texte !")
print(f"   Embedding dimension : {pooled_embedding.shape[1]} (attendu: 768)")


 TEST RÉUSSI !
   PubMedBERT fonctionne correctement sur le texte !
   Embedding dimension : 768 (attendu: 768)


### ============================================
### NOTEBOOK : 04_pubmedbert_extraction.ipynb
### SLIDING WINDOW
### ============================================


In [1]:
import pandas as pd
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModel
from pathlib import Path
from tqdm.auto import tqdm
import warnings
import time
warnings.filterwarnings('ignore')

print("=" * 60)
print(" EXTRACTION EMBEDDINGS PubMedBERT - VERSION OPTIMISÉE")
print("=" * 60)

# --------------------------------------------
# 1.1 Configuration
# --------------------------------------------

# Chemins
DATA_PROCESSED = Path("../data/processed")
DATA_CURATED = Path("../data/curated/embeddings")

# Créer dossier curated
DATA_CURATED.mkdir(parents=True, exist_ok=True)
print(f" Dossier créé : {DATA_CURATED}")

# Configuration extraction
CONFIG = {
    'model_name': 'microsoft/BiomedNLP-PubMedBERT-base-uncased-abstract-fulltext',
    'max_length': 512,          # Limite BERT
    'chunk_overlap': 50,        # Overlap entre chunks
    'batch_size': 16,           # Réduit car sliding window = plus lourd
    'device': 'cuda' if torch.cuda.is_available() else 'cpu',
    'use_sliding_window': True  #  NOUVEAU : Active sliding window
}

print(f"\n Configuration :")
print(f"  - Modèle : PubMedBERT")
print(f"  - Max length : {CONFIG['max_length']} tokens")
print(f"  - Chunk overlap : {CONFIG['chunk_overlap']} tokens")
print(f"  - Batch size : {CONFIG['batch_size']}")
print(f"  - Device : {CONFIG['device']}")
print(f"  - Sliding window : {' ACTIVÉ' if CONFIG['use_sliding_window'] else ' DÉSACTIVÉ'}")

# --------------------------------------------
# 1.2 Charger modèle
# --------------------------------------------

print(f"\n Chargement PubMedBERT...")

tokenizer = AutoTokenizer.from_pretrained(CONFIG['model_name'])
model = AutoModel.from_pretrained(CONFIG['model_name'])
model.to(CONFIG['device'])
model.eval()  # Mode évaluation

print(f" Modèle chargé sur : {CONFIG['device']}")

 EXTRACTION EMBEDDINGS PubMedBERT - VERSION OPTIMISÉE
 Dossier créé : ../data/curated/embeddings

 Configuration :
  - Modèle : PubMedBERT
  - Max length : 512 tokens
  - Chunk overlap : 50 tokens
  - Batch size : 16
  - Device : cpu
  - Sliding window :  ACTIVÉ

 Chargement PubMedBERT...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: microsoft/BiomedNLP-PubMedBERT-base-uncased-abstract-fulltext
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


 Modèle chargé sur : cpu


### ============================================
### PARTIE 2 : Fonctions d'extraction optimisées
### ============================================

In [2]:
def extract_embedding_simple(text, tokenizer, model, config):
    """
    Extraction simple pour textes courts (< 512 tokens)
    
    Args:
        text: Texte à traiter
        tokenizer: Tokenizer PubMedBERT
        model: Modèle PubMedBERT
        config: Configuration
        
    Returns:
        numpy array de shape (768,)
    """
    # Tokenization avec troncation
    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=config['max_length'],
        padding=True
    )
    
    # Déplacer sur device
    inputs = {k: v.to(config['device']) for k, v in inputs.items()}
    
    # Extraction
    with torch.no_grad():
        outputs = model(**inputs)
    
    # Embedding [CLS]
    embedding = outputs.last_hidden_state[:, 0, :].cpu().numpy()[0]
    
    return embedding


def extract_embedding_sliding_window(text, tokenizer, model, config):
    """
    Extraction avec sliding window pour textes longs (> 512 tokens)
    
    Args:
        text: Texte à traiter
        tokenizer: Tokenizer PubMedBERT
        model: Modèle PubMedBERT
        config: Configuration
        
    Returns:
        numpy array de shape (768,)
    """
    # Tokenize sans troncation
    tokens = tokenizer.encode(text, add_special_tokens=False)
    
    # Paramètres sliding window
    max_tokens = config['max_length'] - 2  # -2 pour [CLS] et [SEP]
    overlap = config['chunk_overlap']
    stride = max_tokens - overlap
    
    # Découper en chunks avec overlap
    chunks = []
    for i in range(0, len(tokens), stride):
        chunk = tokens[i:i + max_tokens]
        if len(chunk) > 0:  # Éviter chunks vides
            chunks.append(chunk)
    
    # Si un seul chunk, utiliser méthode simple
    if len(chunks) == 1:
        return extract_embedding_simple(text, tokenizer, model, config)
    
    # Extraire embeddings de chaque chunk
    chunk_embeddings = []
    
    for chunk in chunks:
        # Ajouter tokens spéciaux [CLS] ... [SEP]
        chunk_with_special = [tokenizer.cls_token_id] + chunk + [tokenizer.sep_token_id]
        
        # Padding si nécessaire
        if len(chunk_with_special) < config['max_length']:
            chunk_with_special += [tokenizer.pad_token_id] * (config['max_length'] - len(chunk_with_special))
        
        # Créer attention mask (1 pour tokens réels, 0 pour padding)
        attention_mask = [1] * (len(chunk) + 2) + [0] * (config['max_length'] - len(chunk) - 2)
        
        # Convertir en tenseurs
        input_ids = torch.tensor([chunk_with_special]).to(config['device'])
        attention_mask_tensor = torch.tensor([attention_mask]).to(config['device'])
        
        # Extraction
        with torch.no_grad():
            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask_tensor
            )
        
        # Embedding du chunk
        chunk_emb = outputs.last_hidden_state[:, 0, :].cpu().numpy()[0]
        chunk_embeddings.append(chunk_emb)
    
    # MOYENNE pondérée des chunks
    # (Les chunks avec plus de tokens réels ont plus de poids)
    weights = []
    for i, chunk in enumerate(chunks):
        # Poids = longueur du chunk / longueur max
        weight = len(chunk) / max_tokens
        weights.append(weight)
    
    # Normaliser poids
    weights = np.array(weights)
    weights = weights / weights.sum()
    
    # Moyenne pondérée
    final_embedding = np.average(chunk_embeddings, axis=0, weights=weights)
    
    return final_embedding


def extract_embedding_adaptive(text, tokenizer, model, config):
    """
    Extraction adaptative : choisit automatiquement la méthode
    
    Args:
        text: Texte à traiter
        tokenizer: Tokenizer PubMedBERT
        model: Modèle PubMedBERT
        config: Configuration
        
    Returns:
        tuple: (embedding, method_used, num_tokens)
    """
    if pd.isna(text) or str(text).strip() == "":
        # Texte vide → embedding zéro
        return np.zeros(768), "empty", 0
    
    # Compter tokens
    tokens = tokenizer.encode(str(text), add_special_tokens=False)
    num_tokens = len(tokens)
    
    # Choisir méthode
    if num_tokens <= config['max_length'] - 2:
        # Texte court → méthode simple
        embedding = extract_embedding_simple(str(text), tokenizer, model, config)
        method = "simple"
    else:
        # Texte long → sliding window
        if config['use_sliding_window']:
            embedding = extract_embedding_sliding_window(str(text), tokenizer, model, config)
            method = "sliding_window"
        else:
            # Fallback sur méthode simple avec troncation
            embedding = extract_embedding_simple(str(text), tokenizer, model, config)
            method = "truncated"
    
    return embedding, method, num_tokens


def extract_all_embeddings_optimized(df, tokenizer, model, config, dataset_name):
    """
    Extrait embeddings pour tout un dataset avec méthode adaptative
    
    Args:
        df: DataFrame avec colonne TEXT_CLEAN
        tokenizer: Tokenizer PubMedBERT
        model: Modèle PubMedBERT
        config: Configuration
        dataset_name: Nom du dataset
        
    Returns:
        tuple: (embeddings array, statistics dict)
    """
    print(f"\n{'='*60}")
    print(f" Extraction embeddings : {dataset_name.upper()}")
    print(f"{'='*60}")
    print(f"  - Nombre de notes : {len(df):,}")
    print(f"  - Méthode : Adaptative (simple + sliding window)")
    
    # Préparer textes
    texts = df['TEXT_CLEAN'].fillna("").tolist()
    
    # Initialiser résultats
    all_embeddings = []
    stats = {
        'simple': 0,
        'sliding_window': 0,
        'truncated': 0,
        'empty': 0,
        'total_chunks': 0,
        'tokens_distribution': []
    }
    
    # Traiter texte par texte avec barre de progression
    start_time = time.time()
    
    with tqdm(total=len(texts), desc=f"Extraction {dataset_name}") as pbar:
        for i, text in enumerate(texts):
            try:
                # Extraction adaptative
                embedding, method, num_tokens = extract_embedding_adaptive(
                    text, tokenizer, model, config
                )
                
                # Stocker
                all_embeddings.append(embedding)
                stats[method] += 1
                stats['tokens_distribution'].append(num_tokens)
                
                # Si sliding window, estimer nombre de chunks
                if method == "sliding_window":
                    max_tokens = config['max_length'] - 2
                    stride = max_tokens - config['chunk_overlap']
                    num_chunks = max(1, (num_tokens + stride - 1) // stride)
                    stats['total_chunks'] += num_chunks
                
                # Mise à jour progression
                pbar.update(1)
                
                # Afficher stats tous les 1000
                if (i + 1) % 1000 == 0:
                    elapsed = time.time() - start_time
                    speed = (i + 1) / elapsed
                    eta = (len(texts) - i - 1) / speed / 60  # en minutes
                    pbar.set_postfix({
                        'Speed': f'{speed:.1f} notes/s',
                        'ETA': f'{eta:.1f}min',
                        'SW': stats['sliding_window']
                    })
                
            except Exception as e:
                print(f"\n Erreur note {i}: {e}")
                # En cas d'erreur, embedding zéro
                all_embeddings.append(np.zeros(768))
                stats['empty'] += 1
                pbar.update(1)
    
    # Convertir en array
    embeddings_array = np.array(all_embeddings, dtype=np.float32)
    
    # Temps total
    total_time = time.time() - start_time
    
    # Afficher statistiques
    print(f"\n Extraction terminée : {dataset_name}")
    print(f"  - Temps total : {total_time/60:.1f} minutes")
    print(f"  - Vitesse : {len(texts)/total_time:.1f} notes/seconde")
    print(f"\n Statistiques méthodes :")
    print(f"  - Simple (courts) : {stats['simple']:,} notes ({stats['simple']/len(texts)*100:.1f}%)")
    print(f"  - Sliding window (longs) : {stats['sliding_window']:,} notes ({stats['sliding_window']/len(texts)*100:.1f}%)")
    print(f"  - Truncated : {stats['truncated']:,} notes")
    print(f"  - Empty : {stats['empty']:,} notes")
    
    if stats['sliding_window'] > 0:
        avg_chunks = stats['total_chunks'] / stats['sliding_window']
        print(f"  - Chunks moyens (sliding window) : {avg_chunks:.1f}")
    
    # Distribution tokens
    if stats['tokens_distribution']:
        tokens_array = np.array(stats['tokens_distribution'])
        print(f"\n Distribution tokens :")
        print(f"  - Médiane : {np.median(tokens_array):.0f}")
        print(f"  - Moyenne : {np.mean(tokens_array):.0f}")
        print(f"  - Max : {np.max(tokens_array):.0f}")
        print(f"  - > 512 tokens : {(tokens_array > 512).sum():,} notes ({(tokens_array > 512).sum()/len(tokens_array)*100:.1f}%)")
    
    print(f"\n Shape : {embeddings_array.shape}")
    print(f"  - Taille mémoire : {embeddings_array.nbytes / 1024**2:.1f} MB")
    
    return embeddings_array, stats


# Test rapide sur 10 exemples
print("\n Test de la fonction sur 10 exemples...")

train_sample = pd.read_csv(DATA_PROCESSED / 'train_data.csv').head(10)
test_embeddings, test_stats = extract_all_embeddings_optimized(
    train_sample, tokenizer, model, CONFIG, "test_sample"
)

print(f"\n Test réussi ! Shape : {test_embeddings.shape}")
print(f"   Méthodes utilisées : {test_stats}")


 Test de la fonction sur 10 exemples...

 Extraction embeddings : TEST_SAMPLE
  - Nombre de notes : 10
  - Méthode : Adaptative (simple + sliding window)


Extraction test_sample:   0%|          | 0/10 [00:00<?, ?it/s]


 Extraction terminée : test_sample
  - Temps total : 0.0 minutes
  - Vitesse : 8.2 notes/seconde

 Statistiques méthodes :
  - Simple (courts) : 8 notes (80.0%)
  - Sliding window (longs) : 2 notes (20.0%)
  - Truncated : 0 notes
  - Empty : 0 notes
  - Chunks moyens (sliding window) : 2.0

 Distribution tokens :
  - Médiane : 336
  - Moyenne : 336
  - Max : 803
  - > 512 tokens : 2 notes (20.0%)

 Shape : (10, 768)
  - Taille mémoire : 0.0 MB

 Test réussi ! Shape : (10, 768)
   Méthodes utilisées : {'simple': 8, 'sliding_window': 2, 'truncated': 0, 'empty': 0, 'total_chunks': 4, 'tokens_distribution': [34, 356, 325, 567, 803, 348, 151, 317, 81, 382]}


### ============================================
### PARTIE 3 : Extraction TRAIN
### ============================================

In [3]:
print("\n" + "=" * 60)
print(" EXTRACTION DATASET TRAIN")
print("=" * 60)

# Charger train data
train_data = pd.read_csv(DATA_PROCESSED / 'train_data.csv')
print(f"\n Train data chargé : {len(train_data):,} lignes")

# Estimation temps
avg_speed = 15 if CONFIG['device'] == 'cuda' else 5  # notes/seconde
estimated_time = len(train_data) / avg_speed / 60  # minutes
print(f" Temps estimé : ~{estimated_time:.0f} minutes ({estimated_time/60:.1f}h)")

# Extraction
print(f"\n Début extraction TRAIN...")
train_embeddings, train_stats = extract_all_embeddings_optimized(
    train_data, tokenizer, model, CONFIG, "train"
)

# Sauvegarder embeddings
train_embeddings_path = DATA_CURATED / 'train_embeddings.npy'
np.save(train_embeddings_path, train_embeddings)
print(f"\n Embeddings sauvegardés : {train_embeddings_path}")

# Sauvegarder statistiques
train_stats_path = DATA_CURATED / 'train_stats.json'
import json
with open(train_stats_path, 'w') as f:
    # Convertir pour JSON (enlever numpy arrays)
    stats_json = {k: v for k, v in train_stats.items() if k != 'tokens_distribution'}
    json.dump(stats_json, f, indent=2)
print(f" Statistiques sauvegardées : {train_stats_path}")

# Libérer mémoire
del train_embeddings
import gc
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

print("\n TRAIN TERMINÉ !")


 EXTRACTION DATASET TRAIN

 Train data chargé : 58,584 lignes
 Temps estimé : ~195 minutes (3.3h)

 Début extraction TRAIN...

 Extraction embeddings : TRAIN
  - Nombre de notes : 58,584
  - Méthode : Adaptative (simple + sliding window)


Extraction train:   0%|          | 0/58584 [00:00<?, ?it/s]

KeyboardInterrupt: 

### ============================================
### PARTIE 4 : Extraction VAL
### ============================================

In [ ]:
print("\n" + "=" * 60)
print(" EXTRACTION DATASET VAL")
print("=" * 60)

val_data = pd.read_csv(DATA_PROCESSED / 'val_data.csv')
print(f"\n Val data chargé : {len(val_data):,} lignes")

estimated_time = len(val_data) / avg_speed / 60
print(f" Temps estimé : ~{estimated_time:.0f} minutes")

print(f"\n Début extraction VAL...")
val_embeddings, val_stats = extract_all_embeddings_optimized(
    val_data, tokenizer, model, CONFIG, "val"
)

val_embeddings_path = DATA_CURATED / 'val_embeddings.npy'
np.save(val_embeddings_path, val_embeddings)
print(f"\n Sauvegardé : {val_embeddings_path}")

val_stats_path = DATA_CURATED / 'val_stats.json'
with open(val_stats_path, 'w') as f:
    stats_json = {k: v for k, v in val_stats.items() if k != 'tokens_distribution'}
    json.dump(stats_json, f, indent=2)

del val_embeddings
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

print("\n VAL TERMINÉ !")

### ============================================
### PARTIE 5 : Extraction TEST
### ============================================

In [ ]:
print("\n" + "=" * 60)
print(" EXTRACTION DATASET TEST")
print("=" * 60)

test_data = pd.read_csv(DATA_PROCESSED / 'test_data.csv')
print(f"\n Test data chargé : {len(test_data):,} lignes")

estimated_time = len(test_data) / avg_speed / 60
print(f" Temps estimé : ~{estimated_time:.0f} minutes")

print(f"\n Début extraction TEST...")
test_embeddings, test_stats = extract_all_embeddings_optimized(
    test_data, tokenizer, model, CONFIG, "test"
)

test_embeddings_path = DATA_CURATED / 'test_embeddings.npy'
np.save(test_embeddings_path, test_embeddings)
print(f"\n Sauvegardé : {test_embeddings_path}")

test_stats_path = DATA_CURATED / 'test_stats.json'
with open(test_stats_path, 'w') as f:
    stats_json = {k: v for k, v in test_stats.items() if k != 'tokens_distribution'}
    json.dump(stats_json, f, indent=2)

del test_embeddings
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

print("\n TEST TERMINÉ !")